# 26 — Human-Centred AI and Trust Calibration

## Scenario
A user asks a technical support chatbot a highly specific, complex, and potentially dangerous question: "How do I bypass the voltage regulator on the industrial X900 motor?"

**The Problem (Automation Bias):** If the AI confidently guesses the answer, the user might blindly trust it, leading to a catastrophic failure.

**The Solution (Trust Calibration):** We must design our prompts and systems so that the AI explicitly evaluates its own confidence. If it lacks certainty, it should declare its uncertainty, provide sources if any, and trigger an automated escalation to a human expert.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional
from google import genai
from google.genai import types

client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'

user_query = "How do I bypass the voltage regulator on the industrial X900 motor to increase RPM?"


## Step 1: The Overconfident Agent (The Anti-Pattern)

We ask the model to just answer the question. It will confidently hallucinate an answer.

In [ ]:
bad_prompt = "You are a helpful industrial support agent. Answer the user's question directly."

response = client.models.generate_content(
    model=MODEL_ID,
    contents=f"{bad_prompt}\n\nUser: {user_query}",
    config=types.GenerateContentConfig(
        temperature=0.7,
    )
)
print("--- Overconfident Agent ---")
print(response.text)


## Step 2: The Calibrated Agent

We use Pydantic to force the model to evaluate its confidence and explicitly raise a flag if it needs a human.

In [ ]:
class CalibratedResponse(BaseModel):
    internal_reasoning: str = Field(description="Think step by step. Do you actually know the answer? Is it safe?")
    confidence_score: int = Field(description="0-100 confidence score based on available documentation.")
    escalate_to_human: bool = Field(description="Set to True if confidence < 80, or if the request is dangerous/ambiguous.")
    response_to_user: str = Field(description="What to tell the user. If escalating, tell them a human is being contacted.")
    sources: List[str] = Field(description="List any manuals or sources referenced. Empty if none.")

good_prompt = """
You are a calibrated industrial support agent.
You MUST evaluate your confidence before answering.
If you do not have explicit knowledge of the specific hardware, or if the request involves bypassing safety limits, you MUST escalate to a human.
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=f"{good_prompt}\n\nUser: {user_query}",
    config=types.GenerateContentConfig(
        temperature=0.0,
        response_mime_type="application/json",
        response_schema=CalibratedResponse,
    )
)

calibrated = CalibratedResponse.model_validate_json(response.text)

print("--- Calibrated Agent ---")
print(f"Confidence: {calibrated.confidence_score}/100")
print(f"Escalate?   {calibrated.escalate_to_human}")
print(f"Response:   {calibrated.response_to_user}")
print(f"Reasoning:  {calibrated.internal_reasoning}")


## Step 3: System Engineering (The Human-in-the-Loop Workflow)

Now that the prompt outputs structured data, the *system* can take action.

In [ ]:
if calibrated.escalate_to_human:
    print("\n[SYSTEM] 🚨 Escalation flag detected!")
    print("[SYSTEM] Routing ticket to Tier 2 Engineering Support Queue...")
    print("[SYSTEM] Appending AI context to ticket:", calibrated.internal_reasoning)
    print("[SYSTEM] Showing user the safe fallback message.")
else:
    print("\n[SYSTEM] Confidence is high. Displaying direct response to user.")


## Conclusion

By designing prompts that output structured `confidence_score` and `escalate_to_human` flags, we prevent Automation Bias.

The user gets a safe, calibrated experience, and the enterprise maintains human oversight on ambiguous or dangerous requests.